In [4]:
import random

random_list = [random.random() for _ in range(5_000_000)]

print("Random list length:", len(random_list))

Random list length: 5000000


In [6]:
import numpy as np

np_random_list = np.random.random(5_000_000)

print("Numpy array length:", np_random_list.size)

Numpy array length: 5000000


In [8]:
def standardDeviation(xi, x2, n):
    total = 0
    for x in xi:
        total = total + ((x - x2) * (x - x2))

    deviation = (total / n) ** 0.5

    return deviation

media = sum(random_list) / len(random_list)
print("Desviacion: ", standardDeviation(random_list, media, len(random_list)))

Desviacion:  0.2888206830373956


In [7]:
def standardDeviation(xi, x2, n):
    total = ((xi - x2) ** 2).sum()

    deviation = np.sqrt(total / n)

    return deviation

media = np_random_list.mean()
print("Desviacion: ", standardDeviation(np_random_list, media, np_random_list.size))
print("Comprobacion con np.std: ", np.std(np_random_list))

Desviacion:  0.28876573666051875
Comprobacion con np.std:  0.28876573666051875


In [9]:
import pandas as pd
import numpy as np

datos = pd.Series([10, 20, 30, 40, 50])
print("¡Entorno configurado con exito!")
print(datos.describe())

¡Entorno configurado con exito!
count     5.000000
mean     30.000000
std      15.811388
min      10.000000
25%      20.000000
50%      30.000000
75%      40.000000
max      50.000000
dtype: float64


In [10]:
import pandas as pd
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

print(df.head())
print(df.describe())



   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
  

# Preguntas de análisis

### Pregunta A

Observen la columna `Fare` (Tarifa del boleto). ¿Cuál es la diferencia matemática entre el percentil 75% (el tercer cuartil) y el valor Máximo (max)? Como ingenieros de software, ¿qué les dice esta enorme dispersión estadística sobre la homogeneidad de los precios que se cobraron en el barco?

**Respuesta:** La diferencia es `512.3292 - 31.0000 = 481.3292`. El máximo es 16.5 veces el tercer cuartil y 35.4 veces la mediana (14.45): el 75% de los pasajeros pagó 31 o menos, pero el boleto más caro costó 512.33.

Los precios **no son homogéneos**. La distribución está sesgada a la derecha: casi todos concentrados en tarifas bajas y una cola larga de tarifas altísimas. Esto pasa porque `Fare` mezcla poblaciones distintas en una sola columna (primera, segunda y tercera clase, además de boletos de grupo). Dos consecuencias:

- La media (32.20) no representa al pasajero típico — está arrastrada por los outliers, y la desviación estándar (49.69) es mayor que la media misma. Hay que describir esta columna con mediana y cuartiles.
- Para el modelo, esa cola necesita transformación logarítmica (`np.log1p`) o escalado robusto. Si no, el pasajero de 512 pesaría más que miles de pasajeros de 7.25.

### Pregunta B

Revisen la fila `count` (conteo total de registros válidos) de las distintas columnas en el resultado de su `.describe()`. Comparen el conteo de la columna `PassengerId` con el conteo de la columna `Age`. ¿Qué anomalía de base de datos acaban de descubrir?

**Respuesta:** `PassengerId` tiene 891 registros y `Age` solo 714, así que **faltan 177 edades (19.87%, casi 1 de cada 5 pasajeros)**. La anomalía son valores nulos (`NaN`) en `Age`.

Se detecta así porque el `count` de pandas no cuenta filas sino **valores no nulos**. Como `PassengerId` es la llave primaria y nunca es nulo, sirve de referencia del total real de filas.

`Age` no es la única afectada:

| Columna | Nulos | % |
|---|---|---|
| `Cabin` | 687 | 77.1% |
| `Age` | 177 | 19.9% |
| `Embarked` | 2 | 0.2% |

Además, los faltantes **no son aleatorios**: quien no tiene edad registrada sobrevivió en 29.4%, contra 40.6% de quien sí la tiene. La ausencia del dato es en sí misma información.

### Pregunta C

El algoritmo que construirán al final del semestre requiere que todos los pasajeros tengan un valor numérico en la columna `Age` para poder compilar. Sabiendo lo que acaban de descubrir en la Pregunta B, propongan al menos dos estrategias a nivel de base de datos o de programación para resolver este problema antes de alimentar a la IA.

**Respuesta:**

**1. Imputar con la mediana global**

```python
df["Age"] = df["Age"].fillna(df["Age"].median())   # 28.0
```

La más simple, y conserva toda la muestra. Se usa mediana y no media (29.7) porque es robusta a outliers. Su defecto: mete 177 valores idénticos, lo que reduce artificialmente la varianza.

**2. Imputar con la mediana por grupo**

```python
df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda s: s.fillna(s.median())
)
```

`Pclass` y `Sex` nunca son nulos y están correlacionados con la edad, así que da estimaciones más fieles: 40 para un hombre de 1ª clase vs. 25 para uno de 3ª, en lugar de un solo 28 para todos.

**3. Guardar la bandera del faltante antes de imputar**

```python
df["Age_faltante"] = df["Age"].isnull().astype(int)   # PRIMERO
df["Age"] = df["Age"].fillna(...)                     # DESPUÉS
```

Como en la Pregunta B vimos que la ausencia predice supervivencia, conviene conservar esa señal en lugar de borrarla.

**4. A nivel de base de datos**

- Restricción `NOT NULL` con validación en la capa de ingesta, para que el problema no vuelva a entrar al sistema.
- Guardar el valor estimado en una columna aparte (`Age_imputada`) sin sobrescribir `Age`, para mantener la trazabilidad entre dato medido y dato estimado.
- **Nunca** usar `0` o `-1` como "desconocido": el modelo los leería como edades reales.

> No conviene usar `df.dropna(subset=["Age"])`, porque tiraría el 20% de los datos y, dado que los faltantes se concentran en los que no sobrevivieron, sesgaría el modelo.